# Explore the AutoStrat → EvoMachine pipeline

This notebook sends one natural-language request to an OpenAI-compatible model, validates and semantically verifies the generated DSL, wraps it as an EvoMachine `AbstractStrategy`, and previews the resulting `AutomatonCommand` lists.

It can optionally execute against EvoMachine's virtual peripherals. It never communicates with physical microscope hardware.

In [ ]:
from getpass import getpass
import os
from pathlib import Path

from autostrat import StrategyPipeline, StrategyRequestRejectedError, load_domain_pack
from autostrat.generation import GeneratorConfig, PromptRecipe
from autostrat.verification import SemanticVerifierConfig

from evomachine.coordinates import Coordinate
from evomachine.image_processing_config import ImageProcessorConfigFactory
from evomachine.strategy_generation import (
    MicroscopyCommandAdapter,
    MicroscopyObservationProvider,
    MicroscopyRuntimeErrorProvider,
    StrategyGenerationService,
)
from evomachine.types import LEDType

In [ ]:
def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'evomachine').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from inside the EvoMachine repository.')


repository_root = find_repository_root()
domain = load_domain_pack(repository_root / 'evomachine/domain_packs/microscopy')
print(f'Repository: {repository_root}')
print(f'Domain: {domain.metadata.id} {domain.metadata.version}')
print(f'Commands: {list(domain.commands)}')
print(f'Observations: {list(domain.observations)}')

## Model connection

The defaults below target Robin's OpenAI-compatible endpoint. The key is read from `OPENAI_API_KEY`, or requested without displaying it. Nothing is written to the repository.

In [ ]:
base_url = os.getenv('OPENAI_BASE_URL', 'https://robin-office-2.tail32bb7.ts.net/v1')
model_id = os.getenv('AUTOSTRAT_MODEL_ID', 'qwen3.6')
model = model_id if ':' in model_id else f'openai-chat:{model_id}'

os.environ['OPENAI_BASE_URL'] = base_url
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI-compatible API key: ')

print(f'Endpoint: {base_url}')
print(f'Pydantic AI model: {model}')

## Editable experiment settings

In [ ]:
user_request = (
"""


During initialisation:

For every image acquisition, use LED brightness 10 percent and no filter.

- Move to the first field of view.
- Project the full field for 0.1 s using the 450 nm illumination LED at 5 percent brightness.
- Capture an image using an exposure of 7 ms and the 450 nm LED.
- Wait for 0.25 s.

During every step, first inspect the number of previously completed steps.

If at least 12 steps have already been completed, terminate immediately. Do not move, image, or wait in that branch.

Otherwise, use the following behaviour:

If fewer than 3 steps have been completed:

- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 11 ms and the 450 nm LED.
  - Wait for 0.5 s.
  - Move to the next field of view.
- Otherwise:
  - Capture an image using an exposure of 13 ms and the 515 nm LED.
  - Wait for 0.75 s.
  - Move to the next field of view.

Otherwise, if fewer than 6 steps have been completed:

- Move to the next field of view.
- If the current field-of-view identifier is 0:
  - Capture an image using an exposure of 17 ms and the 565 nm LED.
  - Wait for 1 s.
- Otherwise:
  - If the current field-of-view identifier is greater than 2:
    - Capture an image using an exposure of 19 ms and the 565 nm LED.
    - Wait for 1.25 s.
  - Otherwise:
    - Capture an image using an exposure of 23 ms and the 450 nm LED.
    - Wait for 1.5 s.

Otherwise, if fewer than 9 steps have been completed:

- If the current field-of-view identifier is not 0:
  - Move to the next field of view.
  - Capture an image using an exposure of 29 ms and the 450 nm LED.
  - Wait for 2 s.
- Otherwise:
  - Capture an image using an exposure of 31 ms and the 515 nm LED.
  - Wait for 2.5 s.
  - Move to the next field of view.

Otherwise:

- If fewer than 11 steps have been completed:
  - Move to the next field of view.
  - If the current field-of-view identifier is at least 2:
    - Capture an image using an exposure of 37 ms and the 565 nm LED.
    - Wait for 3 s.
  - Otherwise:
    - Capture an image using an exposure of 41 ms and the 565 nm LED.
    - Wait for 3.5 s.
- Otherwise:
  - If the current field-of-view identifier is 0:
    - Capture an image using an exposure of 43 ms and the 450 nm LED.
    - Wait for 4 s.
  - Otherwise:
    - Capture an image using an exposure of 47 ms and the 450 nm LED.
    - Wait for 4.5 s.
  - Move to the next field of view after completing whichever imaging branch applies.

During finalisation:

- Capture an image using an exposure of 50 ms and the 565 nm LED.
- Wait for 1 s.
- Move to the first field of view.


""")

few_shot_count = 2
validation_retries = 2
semantic_output_retries = 2
semantic_revisions = 2

# Saving and segmentation remain application-owned execution policies.
segment_images = False
save_images = False

print(user_request)

## Run generation, deterministic validation and semantic verification

In [ ]:
pipeline = StrategyPipeline(
    domain,
    generator_config=GeneratorConfig(
        model=model,
        validation_retries=validation_retries,
    ),
    verifier_config=SemanticVerifierConfig(
        model=model,
        output_retries=semantic_output_retries,
    ),
    prompt_recipe=PromptRecipe(
        name='evomachine-notebook',
        few_shot_count=few_shot_count,
    ),
    semantic_revisions=semantic_revisions,
)

cfg = ImageProcessorConfigFactory.default_config(
    channels=[LEDType.LED_450_NM],
    channels_seg=[LEDType.LED_450_NM],
)
adapter = MicroscopyCommandAdapter(
    segment_images=segment_images,
    save_images=save_images,
)

generation_timeout_s = 300
strategy = None
try:
    with StrategyGenerationService(
        pipeline,
        domain=domain,
        command_adapter=adapter,
        observation_provider=MicroscopyObservationProvider(),
        runtime_error_provider=MicroscopyRuntimeErrorProvider(),
    ) as service:
        future = service.submit(user_request, cfg)
        strategy = future.result(timeout=generation_timeout_s)
except StrategyRequestRejectedError as error:
    print('AutoStrat rejected the request:')
    print(error.reason)
    print('Revise user_request before running the remaining cells.')
else:
    print(strategy.source)

## Inspect the accepted result

In [ ]:
verified = strategy.verified
print(f'Accepted after {len(verified.attempts)} candidate(s)')
print(f'Semantic verdict: {verified.verdict}')
print(f'Registered automaton command types: {[item.name for item in strategy.register_automaton_commands()]}')

In [ ]:
prompt = verified.accepted.generated.prompt
for index, message in enumerate(prompt.messages, start=1):
    print(f'\n--- Message {index}: {message.role.upper()} ---\n')
    print(message.content)

## Convert the validated strategy into EvoMachine commands

The FOV mapping below simulates application-owned runtime configuration. The step preview simulates a callback at FOV `0`; no commands are executed.

In [ ]:
fovs = {
    0: Coordinate(0, 0, 0),
    1: Coordinate(100, 0, 0),
}

initialise_commands = strategy.initialise(
    fovs=fovs,
    region_of_interests={fov_id: [] for fov_id in fovs},
    fov_processors={},
    dmd=None,
)
step_commands = strategy.callback(fov_id=0, data=[], errors=[])
finalise_commands = strategy.finalise()

In [ ]:
def describe_command(command):
    details = command.command_args
    if command.command_type.name == 'IMAGE':
        metadata = details['frame_metadata']
        metadata_items = metadata if isinstance(metadata, list) else [metadata]
        details = {
            'frames': [
                {
                    'exposure_ms': item.exposure,
                    'leds': {led.name: brightness for led, brightness in (item.leds or {}).items()},
                    'filter': None if item.filter_wheel is None else item.filter_wheel.name,
                }
                for item in metadata_items
            ],
            'segment': details['segment'],
            'save': details['save'],
        }
    elif command.command_type.name == 'PROJECT':
        details = {
            'illumination_led': details['channel'].name,
            'illumination_brightness': details['brightness'],
            'duration_s': details['duration'],
            'pattern_shape': details['image'].shape,
            'pattern_min': int(details['image'].min()),
            'pattern_max': int(details['image'].max()),
        }
    return {
        'command_id': command.command_id,
        'command_type': command.command_type.name,
        'arguments': details,
    }


for lifecycle, commands in (
    ('initialise', initialise_commands),
    ('step', step_commands),
    ('finalise', finalise_commands),
):
    print(f'\n{lifecycle}:')
    if not commands:
        print('  (no commands)')
    for command in commands:
        print(f'  {describe_command(command)}')

## Execute with the virtual automaton

This runs the generated strategy against EvoMachine's in-memory stage, camera, LEDs and DMD. The virtual automaton currently has one FOV, so `next_fov` wraps back to FOV `0`. A timeout and `finally` cleanup prevent an accidentally non-terminating generated strategy from leaving the worker running.

In [ ]:
from threading import Thread
from time import monotonic, sleep

from evomachine.bindings.binding_types import BindingType
from evomachine.bindings.virtual.peripheralcontroller import VirtualPeripheralController
from evomachine.gui.runtime import build_virtual_automaton
from evomachine.peripherals.filterwheel import FilterWheelConfig, FilterWheelFactory
from evomachine.types import FilterWheelType


virtual_timeout_s = 100
inspection_channel = LEDType.LED_515_NM


def add_virtual_filter_wheel(automaton):
    controller = VirtualPeripheralController()
    controller.initialise()
    filter_wheel = FilterWheelFactory.create(
        FilterWheelConfig(
            binding=BindingType.VIRTUAL,
            available_filters=[
                FilterWheelType.FILTER_465nm,
                FilterWheelType.FILTER_527nm,
                FilterWheelType.FILTER_592nm,
                FilterWheelType.NO_FILTER,
                FilterWheelType.BLOCKING,
            ],
            check_alive=False,
        ),
        peripheral_controllers=controller,
        current_filter_type=FilterWheelType.NO_FILTER,
    )
    filter_wheel.initialise()
    automaton.acq_mngr.filter_wheel = filter_wheel
    return automaton

In [ ]:
virtual_automaton = add_virtual_filter_wheel(build_virtual_automaton())
virtual_automaton.set_strategy(strategy)
execution_errors = []


def run_virtual_automaton():
    try:
        virtual_automaton.run()
    except Exception as error:
        execution_errors.append(error)


worker = Thread(
    target=run_virtual_automaton,
    name='autostrat-virtual-automaton',
    daemon=True,
)
deadline = monotonic() + virtual_timeout_s
completed_by_termination = False
run_summary = {}
latest_image = None

try:
    worker.start()
    virtual_automaton.start_strategy()
    while monotonic() < deadline:
        if execution_errors:
            break
        if virtual_automaton.stopped():
            completed_by_termination = True
            break
        sleep(0.01)

    run_summary = {
        'terminated': completed_by_termination,
        'completed_step_callbacks': strategy.callback_counter,
        'current_fov_id': virtual_automaton.get_fov_id(),
        'last_commands': [describe_command(command) for command in virtual_automaton.last_commands],
    }
    try:
        latest_image = virtual_automaton.get_frame(0, inspection_channel)
    except (KeyError, IndexError):
        latest_image = None
finally:
    virtual_automaton.shutdown()
    worker.join(timeout=5)

if execution_errors:
    raise RuntimeError('The virtual automaton failed.') from execution_errors[0]
if not completed_by_termination:
    raise TimeoutError(
        f'The generated strategy did not terminate within {virtual_timeout_s} seconds.'
    )

print(run_summary)
if latest_image is not None:
    print(
        {
            'latest_image_shape': latest_image.shape,
            'latest_image_min': float(latest_image.min()),
            'latest_image_max': float(latest_image.max()),
        }
    )

## Deterministic runtime-error tests

These tests use a small, deterministically validated strategy and deliberately make one virtual peripheral fail. Every scenario uses a fresh strategy and virtual Automaton so retry counters and stop events cannot leak between tests. The assertions check classification, retry exhaustion and finalisation behavior.

In [ ]:
from autostrat.generation import GeneratedStrategy, Prompt
from autostrat.language.parser import parse_strategy
from autostrat.language.validator import validate_strategy
from autostrat.pipeline import StrategyAttempt, VerifiedStrategy
from autostrat.verification import SemanticVerdict
from evomachine.strategy_generation import AutoStratStrategy
from evomachine.types import AutomatonCommandType

fault_test_source = """initialise
    move_fov(target=first_fov)
step
    if error.device_not_ready:
        abort
    if error.movement_failed:
        retry
    if error.image_acquisition_failed:
        retry
    if error.projection_failed:
        retry
    if error.communication_failed:
        retry
    if error.runtime_failure:
        abort
    if observation.step_count >= 6:
        terminate
    else:
        image(exposure=50ms, led=515nm, led_brightness=10, filter=no_filter)
        project(illumination_led=450nm, illumination_brightness=5, duration=0.1s)
        move_fov(target=next_fov)
finalise
    move_fov(target=first_fov)
"""

def make_fault_test_strategy():
    program = validate_strategy(parse_strategy(fault_test_source), domain)
    generated = GeneratedStrategy(
        source=fault_test_source,
        program=program,
        prompt=Prompt(messages=(), recipe_name='runtime-fault-test'),
    )
    attempt = StrategyAttempt(
        number=1, generated=generated, verdict=SemanticVerdict(accepted=True, issues=())
    )
    accepted = VerifiedStrategy(
        request='Deterministic virtual runtime-fault test.',
        accepted=attempt,
        attempts=(attempt,),
        domain_id=domain.metadata.id,
        domain_version=domain.metadata.version,
    )
    return AutoStratStrategy(
        cfg=cfg,
        verified=accepted,
        domain=domain,
        command_adapter=adapter,
        observation_provider=MicroscopyObservationProvider(),
        runtime_error_provider=MicroscopyRuntimeErrorProvider(),
    )

print(fault_test_source)

In [ ]:
def inject_virtual_fault(automaton, *, target, failures, error_factory):
    state = {'attempts': 0, 'failures_remaining': failures}
    if target == 'move':
        original = automaton.focus_nav.move
    elif target == 'image':
        original = automaton.acq_mngr.take_frame
    elif target == 'project':
        original = automaton._execute_project
    else:
        raise ValueError(f'Unsupported fault target: {target!r}')

    def injected(*args, **kwargs):
        state['attempts'] += 1
        if state['failures_remaining'] > 0:
            state['failures_remaining'] -= 1
            raise error_factory()
        return original(*args, **kwargs)

    if target == 'move':
        automaton.focus_nav.move = injected
    elif target == 'image':
        automaton.acq_mngr.take_frame = injected
    else:
        automaton._execute_project = injected
    return state

def run_fault_scenario(name, *, target, failures, error_factory, expected_error, expected_finalised):
    fault_strategy = make_fault_test_strategy()
    automaton = add_virtual_filter_wheel(build_virtual_automaton())
    automaton.set_strategy(fault_strategy)
    injection = inject_virtual_fault(
        automaton, target=target, failures=failures, error_factory=error_factory
    )
    execution_errors = []

    def run_automaton():
        try:
            automaton.run()
        except Exception as error:
            execution_errors.append(error)

    worker = Thread(target=run_automaton, name=f'fault-test-{name}', daemon=True)
    deadline = monotonic() + 15
    try:
        worker.start()
        automaton.start_strategy()
        while monotonic() < deadline and not automaton.stopped() and not execution_errors:
            sleep(0.01)
        if execution_errors:
            raise RuntimeError(f'{name} escaped the runtime boundary') from execution_errors[0]
        if not automaton.stopped():
            raise TimeoutError(f'{name} did not stop within the test timeout')

        classified = list(fault_strategy.failure_history)
        classified_names = [failure.name for failure in classified]
        retry_attempts = [failure.retry_attempt for failure in classified]
        assert classified_names == [expected_error] * failures
        assert retry_attempts == list(range(failures))
        assert automaton._strategy_is_finalised is expected_finalised
        assert len(automaton.runtime_failure_history) == failures
        if expected_error == 'image_acquisition_failed':
            assert fault_strategy.callback_counter >= 6
        return {
            'scenario': name,
            'outcome': 'terminate/finalise' if expected_finalised else 'abort',
            'injected_attempts': injection['attempts'],
            'classified_errors': classified_names,
            'completed_step_callbacks': fault_strategy.callback_counter,
            'retry_attempts': retry_attempts,
            'diagnostics': [failure.message for failure in classified],
            'automaton_failures': [str(failure) for failure in automaton.runtime_failure_history],
        }
    finally:
        automaton.shutdown()
        worker.join(timeout=5)

In [ ]:
fault_scenarios = (
    dict(name='device not ready aborts immediately', target='move', failures=1,
         error_factory=lambda: RuntimeError('stage is not initialised'),
         expected_error='device_not_ready', expected_finalised=False),
    dict(name='movement retries twice then terminates', target='move', failures=3,
         error_factory=lambda: RuntimeError('stage rejected movement'),
         expected_error='movement_failed', expected_finalised=True),
    dict(name='image failure recovers on retry', target='image', failures=2,
         error_factory=lambda: RuntimeError('camera returned no frame'),
         expected_error='image_acquisition_failed', expected_finalised=True),
    dict(name='image exhaustion continues', target='image', failures=3,
         error_factory=lambda: RuntimeError('camera returned no frame'),
         expected_error='image_acquisition_failed', expected_finalised=True),
    dict(name='projection exhaustion continues', target='project', failures=3,
         error_factory=lambda: RuntimeError('DMD rejected projection'),
         expected_error='projection_failed', expected_finalised=True),
    dict(name='communication retries twice then aborts', target='image', failures=3,
         error_factory=lambda: ConnectionError('camera socket disconnected'),
         expected_error='communication_failed', expected_finalised=False),
)

fault_results = [run_fault_scenario(**scenario) for scenario in fault_scenarios]
for result in fault_results:
    print(f"\n{result['scenario']}")
    for key, value in result.items():
        if key != 'scenario':
            print(f'  {key}: {value}')

In [ ]:
# A miscellaneous failure has no failed command to retry and must abort immediately.
runtime_failure_strategy = make_fault_test_strategy()
runtime_failure_strategy.initialise(
    fovs={0: Coordinate(0, 0, 0)},
    region_of_interests={0: []},
    fov_processors={},
    dmd=None,
)
commands = runtime_failure_strategy.callback(
    fov_id=0, data=[], errors=[RuntimeError('unexpected integration failure')]
)
failure = runtime_failure_strategy.failure_history[-1]
assert failure.name == 'runtime_failure'
assert failure.exception_type == 'RuntimeError'
assert 'unexpected integration failure' in failure.message
assert [command.command_type for command in commands] == [
    AutomatonCommandType.ABORT_STRATEGY
]
print({
    'classified_error': failure.name,
    'diagnostic': failure.message,
    'emitted_commands': [command.command_type.name for command in commands],
})